# Ingestão Streaming com CDF

In [ ]:
import uuid
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
from minio import Minio
from functools import partial
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *


MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM7Class03") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

### UTILS

### Declaring tables and locations

In [ ]:
# BRONZE
table_bronze = "clothes_streaming_bronze"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"
location_checkpoint_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}/_checkpoint"
# SILVER
table_silver = "clothes_streaming_silver"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}"
location_checkpoint_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}/_checkpoint"
# GOLD
table_gold = "clothes_streaming_gold"
location_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}"
location_checkpoint_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}/_checkpoint"

### Destroy environment

In [ ]:
# Configuração do cliente MinIO
minio_client = Minio(
    "minio:9000",  # endpoint do MinIO (ex: localhost:9000)
    access_key=MINIO_ACCESS_KEY,  # sua access key
    secret_key=MINIO_SECRET_KEY,  # sua secret key
    secure=False
)

spark.sql(f"DROP TABLE IF EXISTS {DATABASE}.{table_bronze}")
spark.sql(f"DROP TABLE IF EXISTS {DATABASE}.{table_silver}")
spark.sql(f"DROP TABLE IF EXISTS {DATABASE}.{table_gold}")

# Lista com os caminhos dos buckets
locations = [location_bronze, location_silver, location_gold]

# Loop para remover os objetos de cada local
for loc in locations:
    # Remove o esquema "s3a://" e extrai bucket e prefixo
    loc_no_scheme = loc.replace("s3a://", "")
    bucket, prefix = loc_no_scheme.split("/", 1)
    print(f"Removendo objetos do bucket '{bucket}' com prefixo '{prefix}'")
    
    try:
        # Lista os objetos sob o prefixo (remoção recursiva)
        objects = minio_client.list_objects(bucket, prefix=prefix, recursive=True)
        for obj in objects:
            minio_client.remove_object(bucket, obj.object_name)       
    except S3Error as err:
        print(f"Erro ao limpar o bucket '{bucket}' com prefixo '{prefix}': {err}")


In [ ]:
def merge_into_delta_table(cdf_df, batch_id, path):
    """
    Merges CDC data (cdf_df) into the table,
    using the 'id' and 'yearmonthday' fields as the key.
    If the table does not exist, it is created based on the cdf_df structure.
    
    To avoid the error of multiple rows corresponding to the same key, the source DataFrame
    is preprocessed to eliminate duplicates in the key.
    """
    # Tentar carregar a tabela; se não existir, cria uma tabela vazia
    try:
        layer_table = DeltaTable.forPath(spark, path)
    except Exception as e:
        # Tabela não existe. Criando uma tabela vazia
        empty_df = cdf_df.limit(0)
        empty_df.write.format("delta").mode("overwrite").save(path)
        layer_table = DeltaTable.forPath(spark, path)
    
    # Pré-processamento: eliminar duplicatas na chave 'id' e 'yearmonthday'
    deduped_source = cdf_df.dropDuplicates(["id", "yearmonthday"])
    
    # Executa o MERGE: atualiza registros existentes, insere novos e deleta registros ausentes na fonte
    layer_table.alias("t").merge(
        source=deduped_source.alias("s"),
        condition="t.id = s.id AND t.yearmonthday = s.yearmonthday"
    ).whenMatchedDelete(condition="s._change_type = 'delete'") \
        .whenMatchedUpdateAll(condition="s._change_type <> 'delete'") \
        .whenNotMatchedInsertAll(condition="s._change_type <> 'delete'") \
        .execute()

def read_changes(table_origin, table_destination):

    tables_df = spark.sql(f"SHOW TABLES IN {DATABASE}")
    tables_list = [row.tableName for row in tables_df.collect()]
    
    if table_destination in tables_list:
        print(f"A tabela {table_destination} já existe no database {DATABASE}.")
        # Executa a query sem ORDER BY
        history_df = spark.sql(f"DESCRIBE HISTORY {DATABASE}.{table_origin}")
        
        # Ordena o DataFrame pela coluna "version" de forma decrescente
        latest_version = history_df.orderBy("version", ascending=False).first()["version"]
    
        # Define o startingVersion: pega a versão anterior, mas nunca menor que 0
        starting_version = max(int(latest_version) - 1, 0)
        print(f"Lendo mudanças de {table_origin} do CDC: startingVersion = {starting_version}, latest_version = {latest_version}")
        
        # Se a última versão for maior que o starting_version, usamos o endingVersion; 
        # caso contrário, omitimos essa opção.
        if int(latest_version) > starting_version:
             df_cdf = (
                spark.readStream.format("delta")
                     .option("readChangeFeed", "true")
                     .option("startingVersion", starting_version)
                     .option("endingVersion", latest_version)
                     .table(f"{DATABASE}.{table_origin}")
             )
        else:
             df_cdf = (
                spark.readStream.format("delta")
                     .option("readChangeFeed", "true")
                     .option("startingVersion", starting_version)
                     .table(f"{DATABASE}.{table_origin}")
             )
    else:
        # Se a tabela Silver não existir, lê tudo da Bronze (pode ser o primeiro carregamento completo)
        df_cdf = (
            spark.readStream.format("delta")
                 .option("readChangeFeed", "true")
                 .option("startingVersion", 0)
                 .table(f"{DATABASE}.{table_origin}")
        )

    return df_cdf

## Let's create a new table in the gold layer, from an existing one.

- Schema Enforcement/Evolution

In [ ]:
schema = "id int, data_venda timestamp, produto string, categoria string, quantidade long, preco_unitario double, preco_total double"

In [ ]:
location_raw = f"s3a://staging/streaming/clothes"

In [ ]:
# Exemplo de leitura de dados em streaming (ajuste conforme sua fonte e esquema)
staging_df = (
    spark
    .readStream
    .schema(schema)
    .format("csv")
    .option("header", "true")
    .load(location_raw)  # Caminho dos arquivos CSV
)

In [ ]:
staging_df.printSchema()

<hr style="border: 2px solid black;">

<div style="background-color: #851d86; padding: 20px; border-radius: 8px; align: center">
  <h1 style="color: #fff; font-weight: bold;">JOB BRONZE</h1>
</div>

## INGESTION BATCH DAILY - USING SPARK STREAMING WITH TRIGGER

## Creating the BRONZE with CDC enabled (data change feed)

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze} (
    id int,
    data_venda timestamp,
    produto string,
    categoria string,
    quantidade long,
    preco_unitario double,
    preco_total double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_bronze}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
%%time
(
    staging_df
    .writeStream
    .outputMode("append")  # Adiciona novos registros sem sobrescrever os existentes
    .format("delta")
    .option("path", location_bronze)  # Diretório onde a tabela Delta foi criada
    .option("checkpointLocation", location_checkpoint_bronze)  # Local de checkpoint para garantir a consistência
    .trigger(processingTime="30 seconds")  # Modo micro-batch (processamento periódico)
    .start()
)

print("Streaming of written starting...")

<hr style="border: 2px solid black;">

<div style="background-color: #851d86; padding: 20px; border-radius: 8px; align: center">
  <h1 style="color: #fff; font-weight: bold;">JOB SILVER</h1>
</div>

## Creating the SILVER with CDC enabled (data change feed)

## Reading Changes from Bronze

In [ ]:
df_cdf = read_changes(table_origin=table_bronze, table_destination=table_silver)

## Write Changes into Silver

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_silver} (
    id int,
    data_venda timestamp,
    produto string,
    categoria string,
    quantidade long,
    preco_unitario double,
    preco_total double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_silver}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
%%time
merge_func = partial(merge_into_delta_table, path=location_silver)

print("Streaming of written starting...")

(
    df_cdf
    .writeStream
    .format("delta")
    .foreachBatch(merge_func)
    .option("path", location_silver)
    .option("checkpointLocation", location_checkpoint_silver)
    .trigger(processingTime="30 seconds") # Modo micro-batch (processamento periódico)
    .start()
)

## Exploring more Bronze and Silver tables

### BRONZE

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_bronze} ORDER BY id").show(truncate=False)

In [ ]:
# Query to count total days of month there are on table
spark.sql(f"""
SELECT
    date_format(yearmonthday, 'yyyy-MM') AS month,
    COUNT(DISTINCT yearmonthday) AS total_days
FROM {DATABASE}.{table_bronze} 
GROUP BY date_format(yearmonthday, 'yyyy-MM')
""").show(truncate=False)

In [ ]:
spark.sql(f"SELECT COUNT(*) total_bronze FROM {DATABASE}.{table_bronze}").show(truncate=False)

### SILVER

In [ ]:
spark.sql(f"SELECT COUNT(*) total_silver FROM {DATABASE}.{table_silver}").show(truncate=False)

In [ ]:
# Query to count total days of month there are on table
spark.sql(f"""
SELECT
    date_format(yearmonthday, 'yyyy-MM') AS month,
    COUNT(DISTINCT yearmonthday) AS total_days
FROM {DATABASE}.{table_silver} 
GROUP BY date_format(yearmonthday, 'yyyy-MM')
""").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_silver} ORDER BY id").show(truncate=False)

<hr style="border: 2px solid black;">

<div style="background-color: #851d86; padding: 20px; border-radius: 8px; align: center">
  <h1 style="color: #fff; font-weight: bold;">JOB GOLD</h1>
</div>

## Creating the GOLD with CDC enabled (data change feed), and making data agregation of sales

### Reading Changes from Silver

In [ ]:
df_silver = (
    spark.readStream
    .format("delta")
    .table(f"{DATABASE}.{table_silver}")
)

### Transforming data to the GOLD with Watermark

In [ ]:
agg_stream = (
    df_silver
    .withWatermark("data_venda", "10 seconds")  # Define o limite de atraso de 10 seconds
    .groupBy(
        F.window(F.col("data_venda"), "10 seconds"),  # Janela de agregação de 10 seconds
        F.col("categoria"),
        F.col("yearmonthday")
    )
    .agg(
        F.count("*").alias("total_vendas"),  # Contagem de vendas por categoria e janela
        F.sum("preco_total").alias("receita_total"),  # Soma do valor total das vendas na janela
        F.avg("preco_unitario").alias("preco_medio")  # Preço médio dos produtos vendidos
    )
)

agg_stream = agg_stream.select(["categoria", "total_vendas", "receita_total", "preco_medio", "window", "yearmonthday"])

In [ ]:
agg_stream.printSchema()

### Write into GOLD

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_gold} (
    categoria string,
    total_vendas long,
    receita_total double,
    preco_medio double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_gold}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
%%time
print("Streaming of written starting...")

query = (
    agg_stream
    .writeStream
    .format("delta")
    .option("mergeSchema", "true")
    .outputMode("complete")
    .option("path", location_gold)
    .option("checkpointLocation", location_checkpoint_gold)
    .trigger(processingTime="30 seconds") # Modo micro-batch (processamento periódico)
    .start()
)

In [ ]:
query.status

In [ ]:
spark.sql(f"SELECT COUNT(*) total_gold FROM {DATABASE}.{table_gold}").show(truncate=False)

In [ ]:
# Query to count total days of month there are on table
spark.sql(f"""
SELECT
    date_format(yearmonthday, 'yyyy-MM') AS month,
    COUNT(DISTINCT yearmonthday) AS total_days
FROM {DATABASE}.{table_gold} 
GROUP BY date_format(yearmonthday, 'yyyy-MM')
""").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold} LIMIT 10").show(truncate=False)

<hr style="border: 2px solid black;">

<div style="background-color: #851d86; padding: 20px; border-radius: 8px; align: center">
  <h1 style="color: #fff; font-weight: bold;">DATAVIZ GOLD</h1>
</div>

### Building a BAR with Total sales by category

It takes approximately **1:15 minutes** for the data to be updated in GOLD, and for you to be able to view the new data in the graphs.

In [ ]:
df_gold = spark.sql(f"SELECT * FROM {DATABASE}.{table_gold}")
if df_gold.count() > 0:
    df_pandas = df_gold.select(
        F.col("yearmonthday"), 
        F.col("categoria"), 
        F.col("total_vendas")
    ).toPandas()
    
    # Exibir as colunas disponíveis no DataFrame para verificação
    print("Colunas disponíveis no DataFrame:", df_pandas.columns.tolist())
    
    # Garantindo que as colunas corretas estão sendo usadas
    expected_columns = ["yearmonthday", "categoria", "total_vendas"]
    for col in expected_columns:
        if col not in df_pandas.columns:
            print(f"🚨 Atenção: Coluna '{col}' não encontrada no DataFrame!")
    
    # Se as colunas corretas estiverem disponíveis, proceder com o gráfico
    if all(col in df_pandas.columns for col in expected_columns):
        # Convertendo 'yearmonthday' para formato de data
        df_pandas["yearmonthday"] = pd.to_datetime(df_pandas["yearmonthday"])
        df_pandas["yearmonthday"] = df_pandas["yearmonthday"].apply(lambda x: datetime.strftime(x, '%Y-%m-%d'))
    
        # Agrupando os dados por data e categoria, somando as vendas
        df_pandas_grouped = df_pandas.groupby(["yearmonthday", "categoria"], as_index=False)["total_vendas"].sum()
    
        # Pivotando os dados para gerar um DataFrame adequado para o gráfico
        df_pivot = df_pandas_grouped.pivot_table(index="yearmonthday",
                                                 columns="categoria",
                                                 values="total_vendas",
                                                 aggfunc="sum")
    
        # Criando o gráfico de barras empilhadas
        ax = df_pivot.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="tab10")
    
        # Incluindo os valores das vendas dentro de cada segmento das barras
        # O método bar_label adiciona os rótulos (labels) em cada container de barras.
        for container in ax.containers:
            ax.bar_label(container, fmt='%.0f', label_type='center')
    
        # Configurações do gráfico
        plt.xlabel("Data da Venda")
        plt.ylabel("Total de Vendas")
        plt.title("Vendas por Categoria ao Longo do Tempo")
        plt.xticks(rotation=45)
        plt.legend(title="Categoria")
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.tight_layout()
        plt.show()
    
    else:
        print("🚨 Não foi possível gerar o gráfico. Verifique se as colunas corretas estão no DataFrame.")

### Building a BAR with Total sales by day

In [ ]:
df_gold = spark.sql(f"SELECT * FROM {DATABASE}.{table_gold}")
if df_gold.count() > 0:
    df_pandas = df_gold.select(
        F.col("yearmonthday"), 
        F.col("receita_total"), 
    ).toPandas()
    
    # Exibir as colunas disponíveis no DataFrame para verificação
    print("Colunas disponíveis no DataFrame:", df_pandas.columns.tolist())
    
    # Verificar se as colunas necessárias estão presentes
    expected_columns = ["yearmonthday", "receita_total"]
    for col in expected_columns:
        if col not in df_pandas.columns:
            print(f"🚨 Atenção: Coluna '{col}' não encontrada no DataFrame!")
    
    # Se as colunas necessárias existirem, proceder com o gráfico
    if all(col in df_pandas.columns for col in expected_columns):
        # Converter a coluna 'yearmonthday' para o formato datetime
        df_pandas["yearmonthday"] = pd.to_datetime(df_pandas["yearmonthday"])
        df_pandas["yearmonthday"] = df_pandas["yearmonthday"].apply(lambda x: datetime.strftime(x, '%Y-%m-%d'))
    
        # Agrupar os dados por dia, somando os valores de receita_total
        df_grouped = df_pandas.groupby("yearmonthday", as_index=False)["receita_total"].sum()
    
        # Criar o gráfico de barras
        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.bar(df_grouped["yearmonthday"], df_grouped["receita_total"], color="skyblue")
    
        # Adicionar o valor da venda acima de cada barra
        for bar in bars:
            yval = bar.get_height()
            # Posicionar o texto centralizado sobre a barra, com 2 casas decimais
            ax.text(
                bar.get_x() + bar.get_width() / 2, 
                yval, 
                f'{yval:.2f}', 
                ha='center', 
                va='bottom', 
                fontsize=9,
                color='black'
            )
    
        # Configurações do gráfico
        ax.set_xlabel("Data")
        ax.set_ylabel("Receita Total")
        ax.set_title("Total de Vendas por Dia")
        plt.xticks(rotation=45)
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.tight_layout()
    
        # Exibir o gráfico
        plt.show()
    
    else:
        print("🚨 Não foi possível gerar o gráfico. Verifique se as colunas corretas estão no DataFrame.")